<a href="https://colab.research.google.com/github/essanchristian-maker/DI-Bootcamp/blob/master/Week6_Day4_Daily_Challenge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# ============================================================
# BLOC 1 : Installation des librairies nécessaires
# ============================================================
# peft  : Parameter-Efficient Fine-Tuning (LoRA, etc.)
# datasets : chargement de datasets Hugging Face
# transformers : modèles et tokenizers pré-entraînés
# accelerate : optimisation de l'entraînement
# ============================================================

%pip install peft==0.13.2
%pip install datasets
%pip install transformers==4.44.2
%pip install accelerate

# Création du dossier cache pour sauvegarder le modèle
import os
os.makedirs("cache/working", exist_ok=True)
print("✅ Librairies installées et dossier cache créé")

✅ Librairies installées et dossier cache créé


In [4]:
# ============================================================
# BLOC 2 : Chargement du modèle pré-entraîné et du tokenizer
# ============================================================
# bigscience/bloomz-560m :
#   - Modèle de langage causal (génération de texte)
#   - 560 millions de paramètres (petit = rapide sur Colab)
#   - Pré-entraîné sur textes multilingues
# ============================================================

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "bigscience/bloomz-560m"

print(f"⏳ Chargement du tokenizer '{model_name}'...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

print(f"⏳ Chargement du modèle '{model_name}'...")
foundation_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32   # float32 pour CPU (Colab gratuit)
)

print(f"\n✅ Modèle chargé !")
print(f"Nombre total de paramètres : {sum(p.numel() for p in foundation_model.parameters()):,}")
print(f"\nArchitecture (résumé) :")
print(foundation_model)

⏳ Chargement du tokenizer 'bigscience/bloomz-560m'...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


⏳ Chargement du modèle 'bigscience/bloomz-560m'...

✅ Modèle chargé !
Nombre total de paramètres : 559,214,592

Architecture (résumé) :
BloomForCausalLM(
  (transformer): BloomModel(
    (word_embeddings): Embedding(250880, 1024)
    (word_embeddings_layernorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
    (h): ModuleList(
      (0-23): 24 x BloomBlock(
        (input_layernorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (self_attention): BloomAttention(
          (query_key_value): Linear(in_features=1024, out_features=3072, bias=True)
          (dense): Linear(in_features=1024, out_features=1024, bias=True)
          (attention_dropout): Dropout(p=0.0, inplace=False)
        )
        (post_attention_layernorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (mlp): BloomMLP(
          (dense_h_to_4h): Linear(in_features=1024, out_features=4096, bias=True)
          (gelu_impl): BloomGelu()
          (dense_4h_to_h): Linear(in_feature

In [5]:
# ============================================================
# BLOC 3 : Chargement et preprocessing du dataset
# ============================================================
# Dataset : "Abirate/english_quotes"
# Contient des citations célèbres en anglais
# On prend 10% du split "train" pour accélérer l'entraînement
# ============================================================

from datasets import load_dataset

print("⏳ Chargement du dataset 'english_quotes'...")
data = load_dataset("Abirate/english_quotes", split="train")

# Échantillon de 10% pour accélérer l'entraînement
data = data.shuffle(seed=42).select(range(len(data) // 10))
print(f"✅ Dataset chargé : {len(data)} citations (10% du total)")
print(f"\nAperçu d'une citation :")
print(data[0])

# Tokenisation : convertit le texte en IDs numériques
# que le modèle peut comprendre
# batched=True : traite plusieurs exemples à la fois (plus rapide)
data = data.map(
    lambda samples: tokenizer(
        samples["quote"],
        truncation=True,        # coupe les textes trop longs
        max_length=512,         # longueur max en tokens
        padding="max_length"    # complète avec des tokens spéciaux si trop court
    ),
    batched=True
)

# On ne garde que 5 exemples pour l'entraînement
# (suffisant pour démontrer le fine-tuning sur Colab)
train_sample = data.select(range(10))

print(f"\n✅ Tokenisation terminée")
print(f"Colonnes disponibles : {data.column_names}")
print(f"\nExemple tokenisé (premiers tokens) :")
print(train_sample[0]['input_ids'][:20], "...")

⏳ Chargement du dataset 'english_quotes'...
✅ Dataset chargé : 250 citations (10% du total)

Aperçu d'une citation :
{'quote': "“I don't mind making jokes, but I don't want to look like one.”", 'author': 'Marilyn Monroe', 'tags': ['appearance', 'jokes', 'marilyn-monroe']}

✅ Tokenisation terminée
Colonnes disponibles : ['quote', 'author', 'tags', 'input_ids', 'attention_mask']

Exemple tokenisé (premiers tokens) :
[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3] ...


In [16]:
# ============================================================
# BLOC 4 : Configuration LoRA
# ============================================================
# LoRA ajoute de petites matrices d'adaptation (rang r)
# aux couches d'attention du modèle, sans modifier les
# poids originaux — seules ces matrices sont entraînées
# ============================================================

import peft
from peft import LoraConfig, get_peft_model

# Identifier les modules cibles du modèle BLOOM
# (les couches d'attention qu'on va adapter)
print("Modules du modèle :")
for name, module in foundation_model.named_modules():
    if "query_key_value" in name:
        print(f"  → {name}")
        break

lora_config = LoraConfig(
    r=1,                          # rang des matrices LoRA
                                  # r=1 = très léger, suffisant pour la démo
                                  # en production : r=8, r=16, r=32

    lora_alpha=32,                # facteur d'échelle (généralement = 2*r)
                                  # contrôle l'importance des adaptations LoRA

    target_modules=["query_key_value"],  # couches ciblées dans BLOOM
                                          # = les couches d'attention QKV

    lora_dropout=0.05,            # dropout pour régularisation
                                  # évite l'overfitting sur peu de données

    bias="none",                  # on n'entraîne pas les biais
    task_type="CAUSAL_LM"         # tâche : génération de texte causal
)

# Application de LoRA au modèle de base
peft_model = get_peft_model(foundation_model, lora_config)

print("\n✅ LoRA configuré et appliqué !")
print("\nParamètres entraînables vs total :")
peft_model.print_trainable_parameters()
# → attendu : ~0.1% des paramètres seulement !

Modules du modèle :
  → transformer.h.0.self_attention.query_key_value

✅ LoRA configuré et appliqué !

Paramètres entraînables vs total :
trainable params: 98,304 || all params: 559,312,896 || trainable%: 0.0176


In [17]:
# ============================================================
# BLOC 5 : Configuration et lancement de l'entraînement (Version Stable)
# ============================================================

import transformers
from transformers import TrainingArguments, Trainer
import os

output_directory = os.path.join("cache/working", "peft_lab_outputs")
os.makedirs(output_directory, exist_ok=True)

training_args = TrainingArguments(
    report_to="none",
    output_dir=output_directory,
    auto_find_batch_size=True,
    learning_rate=5e-5,           # LR plus faible pour la stabilité
    num_train_epochs=5,            # Réduit à 5 pour éviter le sur-apprentissage sur 10 exemples
    logging_steps=1,
    weight_decay=0.01
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_sample,
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

print("⏳ Entraînement en cours (Paramètres stabilisés)...")
trainer.train()
print("\n✅ Entraînement terminé !")

⏳ Entraînement en cours (Paramètres stabilisés)...


Step,Training Loss


Step,Training Loss
1,3.595900
2,3.686500
3,4.019600
4,3.950700
5,3.407800
6,3.588800
7,3.376400
8,3.903300
9,3.263300
10,3.306700



✅ Entraînement terminé !


In [18]:
# ============================================================
# BLOC 6 : Sauvegarde du modèle fine-tuné
# ============================================================
# On sauvegarde UNIQUEMENT les adaptateurs LoRA
# (pas les 560M paramètres originaux — beaucoup plus léger !)
# ============================================================

import time

time_now = int(time.time())       # timestamp unique pour nommer le dossier
peft_model_path = os.path.join(
    output_directory,
    f"peft_model_{time_now}"
)

trainer.model.save_pretrained(peft_model_path)

print(f"✅ Modèle LoRA sauvegardé dans : {peft_model_path}")
print(f"\nFichiers sauvegardés :")
for f in os.listdir(peft_model_path):
    size = os.path.getsize(os.path.join(peft_model_path, f))
    print(f"  {f} ({size/1024:.1f} KB)")
# → adapter_model.bin (très petit ~quelques KB vs ~1GB pour le modèle complet)

✅ Modèle LoRA sauvegardé dans : cache/working/peft_lab_outputs/peft_model_1782488427

Fichiers sauvegardés :
  README.md (5.0 KB)
  adapter_config.json (0.6 KB)
  adapter_model.safetensors (390.8 KB)


In [19]:
# ============================================================
# BLOC 7 : Chargement du modèle sauvegardé + Inférence
# ============================================================
# On recharge le modèle de base + les adaptateurs LoRA
# is_trainable=False : mode inférence uniquement
# ============================================================

from peft import PeftModel

print("⏳ Rechargement du modèle pour l'inférence...")

# Recharger le modèle de base
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32
)

# Ajouter les adaptateurs LoRA fine-tunés par-dessus
loaded_model = PeftModel.from_pretrained(
    base_model,
    peft_model_path,
    is_trainable=False            # mode inférence : pas d'entraînement
)

print("✅ Modèle LoRA rechargé avec succès !\n")

# ------------------------------------------------------------
# Génération de texte avec le modèle fine-tuné
# ------------------------------------------------------------
prompts = [
    "Two things are infinite: ",
    "The purpose of life is ",
    "In the middle of difficulty "
]

for prompt in prompts:
    inputs = tokenizer(prompt, return_tensors="pt")

    outputs = loaded_model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=50,          # nombre max de tokens à générer
        temperature=0.7,            # créativité (0=déterministe, 1=créatif)
        do_sample=True,             # sampling aléatoire (plus varié)
        pad_token_id=tokenizer.eos_token_id
    )

    generated = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    print(f"Prompt  : '{prompt}'")
    print(f"Réponse : '{generated[0]}'")
    print("-" * 60)

⏳ Rechargement du modèle pour l'inférence...
✅ Modèle LoRA rechargé avec succès !

Prompt  : 'Two things are infinite: '
Réponse : 'Two things are infinite:  the number of individuals and the number of variables.'
------------------------------------------------------------
Prompt  : 'The purpose of life is '
Réponse : 'The purpose of life is  to live quality'
------------------------------------------------------------
Prompt  : 'In the middle of difficulty '
Réponse : 'In the middle of difficulty  there is a drop in water'
------------------------------------------------------------
